### ------------------- Model Used: Pegasus cnn dailymail ----------------

In [10]:
from transformers import pipeline, set_seed
import matplotlib.pyplot as plt 
from datasets import load_dataset
import pandas as pd 
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import nltk
from tqdm import tqdm
import torch
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\tipto\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [11]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [12]:
# initilize the tokenizer
model_checkpoint = "google/pegasus-cnn_dailymail"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

tokenizer_config.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

In [13]:
# load the model
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint).to(device)

pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-cnn_dailymail and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

In [14]:
model

PegasusForConditionalGeneration(
  (model): PegasusModel(
    (shared): Embedding(96103, 1024, padding_idx=0)
    (encoder): PegasusEncoder(
      (embed_tokens): Embedding(96103, 1024, padding_idx=0)
      (embed_positions): PegasusSinusoidalPositionalEmbedding(1024, 1024)
      (layers): ModuleList(
        (0-15): 16 x PegasusEncoderLayer(
          (self_attn): PegasusAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
          (final_layer_no

In [16]:
# load the dataset
dataset = load_dataset("knkarthick/samsum")

README.md: 0.00B [00:00, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/14731 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/818 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/819 [00:00<?, ? examples/s]

In [17]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})

In [22]:
print(dataset['train'][1]['dialogue'])
print(' - ' * 20)
print(dataset['train'][1]['summary'])

Olivia: Who are you voting for in this election? 
Oliver: Liberals as always.
Olivia: Me too!!
Oliver: Great
 -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  - 
Olivia and Olivier are voting for liberals in this election. 


In [24]:
# check how base model is performing
dialouge = dataset['test'][0]['dialogue']
print(dialouge)

Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye


In [26]:
pipe = pipeline(
    task = "summarization" , model = model_checkpoint
)

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-cnn_dailymail and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0


In [27]:
output = pipe(dialouge)
print(output)

Your max_length is set to 128, but your input_length is only 122. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=61)


[{'summary_text': "Amanda: Ask Larry Amanda: He called her last time we were at the park together .<n>Hannah: I'd rather you texted him .<n>Amanda: Just text him ."}]


In [29]:
print(output[0]['summary_text'].replace(" .<n>" , "\n"))

Amanda: Ask Larry Amanda: He called her last time we were at the park together
Hannah: I'd rather you texted him
Amanda: Just text him .


In [30]:
import evaluate
rouge = evaluate.load("rouge")

def compute_rouge(preds , refs):
    return rouge.compute(
        predictions = preds,
        references = refs,
        use_stemmer = True
    )

In [32]:
dataset['test'][0]['summary']

"Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry."

In [33]:
compute_rouge(
    preds = [output[0]['summary_text']] , 
    refs = [dataset['test'][0]['summary']]
)

{'rouge1': 0.13636363636363635,
 'rouge2': 0.0,
 'rougeL': 0.09090909090909091,
 'rougeLsum': 0.09090909090909091}

### As base model is not performing well so we have to do finetuning using our dataset.


### Text Normalization and formating.

In [34]:
import re 
def normalize_dialogue(text: str):
    """Normalize SAMSum dialogue for abstractive summarization."""
    # replace <file> with special tokens
    text = re.sub(r"<file_gif>", "[GIF]", text)
    text = re.sub(r"<file_image>", "[IMAGE]", text)
    text = re.sub(r"<file_video>", "[VIDEO]", text)
    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()
    # Ensure each speaker turn is on a new line
    text = re.sub(r"([A-Za-z]+:)", r"\n\1", text)
    text = text.strip()
    # Task prefix (helps PEGASUS)
    text = "summarize dialogue:\n" + text
    return text 

In [35]:
def normalize_summary(text: str) -> str:
    """
    Normalize SAMSum summaries.
    """
    # Replace file placeholders if present
    text = re.sub(r"<file_gif>", "[GIF]", text)
    text = re.sub(r"<file_image>", "[IMAGE]", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [36]:
def preprocessing(examples):
    # Normalize inputs
    inputs = [normalize_dialogue(d) for d in examples["dialogue"]]
    targets = [normalize_summary(s) for s in examples["summary"]]

    # Tokenize inputs
    model_inputs = tokenizer(
        inputs,
        max_length = 1024,
        truncation = True,
        padding = False
    )

    # Tokenize targets (labels)
    labels = tokenizer(
        text_target = targets,
        max_length = 128,
        truncation = True,
        padding = False
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [37]:
tokenized_dataset = dataset.map(
    preprocessing,
    batched = True,
    remove_columns = dataset["train"].column_names
)

Map:   0%|          | 0/14731 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

In [38]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 818
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 819
    })
})

In [39]:
print(tokenized_dataset['train'][0])

{'input_ids': [24710, 5762, 151, 12195, 151, 125, 7091, 3659, 107, 842, 119, 245, 181, 152, 10508, 151, 7435, 147, 12195, 151, 125, 131, 267, 650, 119, 3469, 29344, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [12195, 7091, 3659, 111, 138, 650, 10508, 181, 3469, 107, 1]}
